In [87]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
import spacy
import re
import os 
import string 

In [88]:
data = PyPDFLoader("AIandMl.pdf").load()

In [89]:
data1 = ""
for i in range(len(data)):
    data1 += data[i].page_content.lower()
data = data1

## Text Normalization

In [90]:
var = string.punctuation
data = re.sub(f'[{var}]',' ',data) 
data = re.sub(r'\s{2,}',' ',data)
# for reducing the tokens

In [91]:
nlp = spacy.load('en_core_web_sm')
tokens = nlp(data)
type(tokens)

spacy.tokens.doc.Doc

In [92]:
updated_tokens = [str(token.lemma_) for token in tokens if not token.is_stop] # is_stop--> returns boolean if the word is stopword
data = " ".join(updated_tokens)
print(data)
# print(updated_tokens)

artificial intelligence machine learn practical introduction concept method application responsible use 10 page learn guide 
 guide explain ai system work machine learning model learn datum technique commonly build project responsibly design student beginner aspire datum ai practitioner ai machine learning learn guide 
 page 2 10 
 1 artificial intelligence artificial intelligence ai broad field computing focus building system perform task 
 associate human intelligence task include understand language recognize image planning action solve problem make recommendation ai mean computer think exactly like person practice ai system use program rule statistical method learn pattern combination produce useful output common category ai 
 ● 
 rule base system follow explicit instruction write developer troubleshooting flow 
 map answer outcomes ● 
 machine learning learn pattern example instead rely manually write rule ● 
 deep learning use neural network multiple processing layer learn comple

In [93]:
# Fixed size Chuncking
# chunk_size = 150
# s2 = []
# for index in range(0,len(data),chunk_size):
#     chunk = data[index:index+chunk_size]
#     s2.append(chunk)
# print(s2)

In [94]:
# chunk_size = 15
# chunk_overlap = 5
# s2 = []
# for index in range(0,len(data),chunk_size-chunk_overlap):
#     chunk = data[index:index+chunk_size]
#     s2.append(chunk)
# print(s2)

In [95]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 100,
    separators=['\n\n','.','!','\n',' ','']
)
chunks = splitter.create_documents([data])
for chunk in chunks:
    print(chunk.page_content)

artificial intelligence machine learn practical introduction concept method application responsible use 10 page learn guide
guide explain ai system work machine learning model learn datum technique commonly build project responsibly design student beginner aspire datum ai practitioner ai machine learning learn guide
page 2 10 
 1 artificial intelligence artificial intelligence ai broad field computing focus building system perform task
associate human intelligence task include understand language recognize image planning action solve problem make recommendation ai mean computer think exactly like person practice ai system use
solve problem make recommendation ai mean computer think exactly like person practice ai system use program rule statistical method learn pattern combination produce useful output common category ai
● 
 rule base system follow explicit instruction write developer troubleshooting flow 
 map answer outcomes ● 
 machine learning learn pattern example instead rely man

In [96]:
chunks = splitter.create_documents([data])
chunks[0].metadata={'file_name' : 'rag.txt'}
chunks

[Document(metadata={'file_name': 'rag.txt'}, page_content='artificial intelligence machine learn practical introduction concept method application responsible use 10 page learn guide'),
 Document(metadata={}, page_content='guide explain ai system work machine learning model learn datum technique commonly build project responsibly design student beginner aspire datum ai practitioner ai machine learning learn guide'),
 Document(metadata={}, page_content='page 2 10 \n 1 artificial intelligence artificial intelligence ai broad field computing focus building system perform task'),
 Document(metadata={}, page_content='associate human intelligence task include understand language recognize image planning action solve problem make recommendation ai mean computer think exactly like person practice ai system use'),
 Document(metadata={}, page_content='solve problem make recommendation ai mean computer think exactly like person practice ai system use program rule statistical method learn pattern 

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-miniLM-L6-V2'
)
embedding_model

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HuggingFaceEmbeddings(model_name='sentence-transformers/all-miniLM-L6-V2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [ ]:
vectordb = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)
vectordb # it has chunks and vectors

In [99]:
user_query = "What is Machine Learning ?"
r_chunks = vectordb.similarity_search(user_query)  
r_chunks

[Document(id='5f10b4bb-04bb-4de5-8e5c-9b3149c1a5d2', metadata={}, page_content='sophisticated ai machine learning learn guide \n page 4 10 \n 3 main learning approach \n ml approach group type feedback available learn 1 supervise learning'),
 Document(id='8085cd0f-f396-42f1-a85a-1c20d993b244', metadata={}, page_content='guide explain ai system work machine learning model learn datum technique commonly build project responsibly design student beginner aspire datum ai practitioner ai machine learning learn guide'),
 Document(id='8796725e-34cb-4b43-9fbb-595b6e06441b', metadata={}, page_content='page 3 10 \n 2 machine learn machine learning ml branch ai algorithms use datum identify pattern prediction \n decision write separate rule possible case developer provide example'),
 Document(id='1abc14b6-d76a-446e-83db-5b19bb14bea0', metadata={}, page_content='method train configure model evaluate deploy real workflow monitor performance \n time key idea ai umbrella field machine learning importa

In [100]:
ret_data = ""
for i in r_chunks:
    ret_data += i.page_content
print(ret_data) 

sophisticated ai machine learning learn guide 
 page 4 10 
 3 main learning approach 
 ml approach group type feedback available learn 1 supervise learningguide explain ai system work machine learning model learn datum technique commonly build project responsibly design student beginner aspire datum ai practitioner ai machine learning learn guidepage 3 10 
 2 machine learn machine learning ml branch ai algorithms use datum identify pattern prediction 
 decision write separate rule possible case developer provide examplemethod train configure model evaluate deploy real workflow monitor performance 
 time key idea ai umbrella field machine learning important approach ai machine learning learn guide 
 page 3 10


In [101]:
def retrieval(query : str,k):
    r_chunk = vectordb.similarity_search(query,k)
    retrieval = [chunk.page_content for chunk in r_chunk]
    ret_text = ' '.join(retrieval)
    return ret_text

user_query = "What is Machine Learning ?"
try:
    retrieval(user_query,3)
except Exception as e:
    print(f"Error : {e}")
finally:
    print(retrieval(user_query,3))

sophisticated ai machine learning learn guide 
 page 4 10 
 3 main learning approach 
 ml approach group type feedback available learn 1 supervise learning guide explain ai system work machine learning model learn datum technique commonly build project responsibly design student beginner aspire datum ai practitioner ai machine learning learn guide page 3 10 
 2 machine learn machine learning ml branch ai algorithms use datum identify pattern prediction 
 decision write separate rule possible case developer provide example


In [ ]:
user_query = "What is Machine Learning?"
def generate_text(query,answer):
    model_gemini = ChatGoogleGenerativeAI(
        model = 'gemini-3.5-flash',
        api_key = os.environ['Gemini_api'],
        temperature = 0
    )
    prompt = f'''
            You're a helpful assistant and structure my given context
            Input : {query}
            Output : {answer}
            Note : 
            1)don't add any extra content.
            2)Whatever there in my {answer} just structure it.
            3)Add the correct words to it         
            Output Format:
            Input : {query}
            Output :  
            '''
    gemini = model_gemini.invoke(prompt)
    print("From Gemini : \n",gemini.content[0]['text'])

generate_text(user_query,retrieval(user_query,3))

From Gemini : 
 Input : What is Machine Learning?
Output :  
**Sophisticated AI Machine Learning Learning Guide**

**Page 3 of 10**
* **2. Machine Learning:** Machine Learning (ML) is a branch of AI where algorithms use data to identify patterns and make predictions or decisions, rather than writing separate rules for every possible case where a developer provides examples.

**Page 4 of 10**
* **3 Main Learning Approaches:** ML approaches are grouped by the type of feedback available to learn.
  1. **Supervised Learning**

**Guide Details**
* **Purpose:** This guide explains how AI systems work, how machine learning models learn from data, and techniques commonly used to build projects responsibly.
* **Target Audience:** Designed for students, beginners, and aspiring data and AI practitioners.


: 